Qui uso il conterfactual fairness dataset.

In [ ]:
!pip install -U huggingface_hub

In [ ]:
!hf auth login --token 

In [ ]:
!pip install --upgrade transformers accelerate bitsandbytes

In [ ]:
import pandas as pd
from transformers import AutoTokenizer, AutoModelForCausalLM, BitsAndBytesConfig
import numpy as np
import json
import random
import itertools

In [ ]:
!gdown 1g9mUG3lo6GDi9n5adjlj0XcZ1hcs9myI

In [ ]:
df=pd.read_csv('/content/counterfactual_fairness_dataset_expanded.csv')
df.head()

In [ ]:
label_cols = [
    "Ground truth Hate",
    "Ground truth Sexual",
    "Ground truth Toxicity",
    "Ground truth Violence",
]

print(df[label_cols].sum().sort_values(ascending=False))

co_occurrence = pd.DataFrame(index=label_cols, columns=label_cols, dtype=int)
for l1, l2 in itertools.product(label_cols, repeat=2):
    co_occurrence.loc[l1, l2] = (df[l1] & df[l2]).sum()

print("=== Co-occurrences ===")
print(co_occurrence)


In [ ]:
print("\n=== Prevalenza (%) per subgroup ===")
prevalence = df.groupby("subgroup")[label_cols].mean().mul(100).round(1)
print(prevalence.to_string())

print("\n=== N sample per subgroup ===")
print(df.groupby("subgroup").size().sort_values(ascending=False))

In [ ]:
output = {
    "label_totals": df[label_cols].sum().to_dict(),
    "co_occurrence": co_occurrence.to_dict(orient="index"),
    "prevalence_by_subgroup": prevalence.to_dict(orient="index"),
    "sample_counts_by_subgroup": df.groupby("subgroup").size().to_dict(),
}

with open("CFD_stats.json", "w", encoding="utf-8") as f:
    json.dump(output, f, indent=2, ensure_ascii=False)

In [ ]:
print(df["subgroup"].unique())

target_prefixes = ["GenderId", "RaceEthnicity", "Religion", "SexualOrien"]

other_labels = [
    "Ground truth Sexual",
    "Ground truth Toxicity",
    "Ground truth Violence",
]

df_hate = df[df["Ground truth Hate"] == 1].copy()
df_hate["label_count"] = df_hate[other_labels].sum(axis=1)

samples_per_target = []

for prefix in target_prefixes:
    df_target = df_hate[df_hate["subgroup"].str.startswith(prefix, na=False)]
    df_target_sorted = df_target.sort_values("label_count", ascending=False)

    top25 = df_target_sorted.head(25).copy()

    print(f"\n=== {prefix} ===")
    print(f"Disponibili: {len(df_target)} | Selezionati: {len(top25)}")
    print(f"Distribuzione label_count:\n{top25['label_count'].value_counts().sort_index(ascending=False)}")

    samples_per_target.append(top25)

sample = pd.concat(samples_per_target, ignore_index=False)
sample = sample.drop(columns=["label_count"])

print(f"\nTotale righe: {len(sample)}")
sample.to_csv("sampled_rows_CFD.csv", index=False)

In [ ]:
texts=sample["prompt"]
texts

In [ ]:
texts.reset_index(drop=True, inplace=True)
texts

In [ ]:
quantization_config = BitsAndBytesConfig(load_in_4bit=True)

model_name_1="meta-llama/Llama-3.1-8B-Instruct"

tokenizer_1 = AutoTokenizer.from_pretrained(model_name_1)
model_1 = AutoModelForCausalLM.from_pretrained(
    model_name_1,
    device_map="auto",
    quantization_config=quantization_config
  )

In [ ]:
prompt = [
    {
        "role": "system",
        "content": "You are a data annotator specialized in hate speech detection."
    },
    {
        "role": "user",
        "content": """Read the provided TEXT and extract the exact substring that correspond to the following Hate Speech components.
        If a component is not present in the text, you must return the string "absent".

        Below you find the components to identify in the given text:

        - Hate: Negative or hateful comments targeting someone due to their identity.
        - Toxicity: A rude, disrespectful, or unreasonable comment that is likely to make people leave a discussion.
        - Sexual: Contains references to sexual acts, body parts, or other lewd content.
        - Violence: Describes an intention to inflict pain, injury, or violence against an individual or group.
        - Race/Ethnicity: Black, Asian, White, LatinX, Indigenous, Biracial
        - Religion: Atheism, Christianity, Hinduism, Islam, Judaism, Buddhism, Others
        - Sexual Orientation: Heterosexual, NonHeterosexual
        - Gender Identity: Male, Female, NonCisgender


        RULES:
        1- You must extract exact quotes from the text. Do not paraphrase or change any words.
        2- Do not output any conversational text, greetings, introductions, or explanations.
        3- Your response must be only the raw json object. Do not wrap the json in markdown formatting.

        OUTPUT FORMAT:
        {
            "Hate": "extracted substring or absent",
            "Toxicity": "extracted substring or absent",
            "Sexual": "extracted substring or absent",
            "Violence": "extracted substring or absent",
            "Race/Ethnicity": "extracted substring or absent",
            "Religion": "extracted substring or absent",
            "Sexual Orientation": "extracted substring or absent",
            "Gender Identity": "extracted substring or absent"
        }

        TEXT: "{text}"

        ANSWER:

        """
    }

]

In [ ]:
def prepare_prompts(texts, prompt_template, tokenizer):
  """
    This function format input text samples into instructions prompts.

    Inputs:
      texts: input texts to classify via prompting
      prompt_template: the prompt template provided in this assignment
      tokenizer: the transformers Tokenizer object instance associated
      with the chosen model card

    Outputs:
      input texts to classify in the form of instruction prompts
  """
  formatted_prompts=[]

  for text in texts:
    prompt_copy = [
            {"role": prompt_template[0]["role"], "content": prompt_template[0]["content"]},
            {
                "role": prompt_template[1]["role"],
                "content": prompt_template[1]["content"].replace("{text}", text)
            }
        ]


    formatted_prompt = tokenizer.apply_chat_template(
      prompt_copy,
      add_generation_prompt=True,
      tokenize=True,
      return_dict=True,
      return_tensors="pt",
    )

    formatted_prompts.append(formatted_prompt)


  return formatted_prompts

In [ ]:
def generate_responses(model, prompt_examples, tokenizer, verbose=False):
  """
    This function implements the inference loop for a LLM model.
    Given a set of examples, the model is tasked to generate
    a response.

    Inputs:
      model: LLM model instance for prompting
      prompt_examples: pre-processed text samples

    Outputs:
      generated responses
  """
  outputs=[]
  for index, prompt_example in enumerate(prompt_examples):
    if verbose and index % 10 == 0:
      print(f"generating response n.{index}...")
    prompt_example=prompt_example.to(model.device)
    output = model.generate(**prompt_example, max_new_tokens=500,pad_token_id=tokenizer.eos_token_id)
    output = tokenizer.decode(output[0][prompt_example["input_ids"].shape[-1]:], skip_special_tokens=True)
    outputs.append(output)

  return outputs

In [ ]:
outputs1 = generate_responses(model_1, prepare_prompts(texts, prompt, tokenizer_1), tokenizer_1, verbose=True)
#20 minuti

In [ ]:
parsed_outputs = []
json_errors = []
for output in outputs1:
    try:
        parsed_outputs.append(json.loads(output))
    except json.JSONDecodeError:
        json_errors.append(output)

with open("results_CFD_Llama.json", "w", encoding="utf-8") as f:
    json.dump(parsed_outputs, f, ensure_ascii=False, indent=2)

with open("json_errors_CFD_Llama.txt", "w", encoding="utf-8") as f:
    f.write("\n".join(json_errors))